# 01: Download Model Weights

**Run once.** Pulls LLaVA-1.6-7B and Mistral-7B-Instruct from HuggingFace into
`/kaggle/working/hf_cache`. After this notebook finishes, click **Save Version**
(Quick Save → 'Save & Run All') and the resulting Kaggle Dataset (named
`gvla-weights`) becomes the read-only weight cache for every other notebook.

## Prerequisites

1. Accept the model licenses on HuggingFace:
   - https://huggingface.co/llava-hf/llava-v1.6-mistral-7b-hf
   - https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.2
2. Add your HF token as a Kaggle Secret named `HF_TOKEN`
   (Add-ons → Secrets → Add a secret).
3. In **Settings** (right panel): Accelerator = `GPU T4 x2`, Persistence = `Files only`,
   Internet = `On`.

Approximate runtime: 8–12 minutes. Approximate disk usage: ~28 GB.

In [ ]:
# Pre-flight: GPU + disk + internet
import shutil, subprocess, os
print('--- GPU ---')
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'])
print('--- Disk (/kaggle/working) ---')
total, used, free = shutil.disk_usage('/kaggle/working')
print(f'free: {free/1e9:.1f} GB / total: {total/1e9:.1f} GB')
assert free > 30 * 1e9, 'need ~30 GB free; restart the session'

In [ ]:
!pip install -q --upgrade huggingface_hub

In [ ]:
# Authenticate with the HF token stored as a Kaggle Secret.
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

token = UserSecretsClient().get_secret('HF_TOKEN')
login(token=token, add_to_git_credential=False)
print('HF auth OK')

In [ ]:
from pathlib import Path
from huggingface_hub import snapshot_download

CACHE = Path('/kaggle/working/hf_cache')
CACHE.mkdir(parents=True, exist_ok=True)

print('Downloading LLaVA-1.6-7B (~13 GB) ...')
snapshot_download(
    repo_id='llava-hf/llava-v1.6-mistral-7b-hf',
    local_dir=CACHE / 'llava-v1.6-mistral-7b-hf',
    local_dir_use_symlinks=False,
    # Drop the redundant .bin shards if .safetensors are present.
    ignore_patterns=['*.bin', '*.msgpack', '*.h5'],
)
print('LLaVA done.')

In [ ]:
print('Downloading Mistral-7B-Instruct (~14 GB) ...')
snapshot_download(
    repo_id='mistralai/Mistral-7B-Instruct-v0.2',
    local_dir=CACHE / 'Mistral-7B-Instruct-v0.2',
    local_dir_use_symlinks=False,
    ignore_patterns=['*.bin', '*.msgpack', '*.h5'],
)
print('Mistral done.')

In [ ]:
# Verify and report.
import subprocess
subprocess.run(['du', '-sh', '/kaggle/working/hf_cache'])
for sub in sorted(CACHE.iterdir()):
    print(sub.name)

## Next step

1. Click **Save Version** → **Quick Save** (or **Save & Run All**) at the top right.
2. After the version is saved, open the notebook's **Output** tab and click
   **New Notebook** → name the dataset `gvla-weights`. (Or, more robust: use
   *Output* → *New Dataset* directly.)
3. Subsequent notebooks mount this dataset read-only at
   `/kaggle/input/gvla-weights/hf_cache`.

**You will not need to re-run this notebook unless model versions change.**